# Mixed Precision Training & torch.compile

Modern GPUs have dedicated hardware for **FP16/BF16** arithmetic — up to 3× faster and half the memory of FP32.  
PyTorch 2.0 added **`torch.compile()`** — graph-level JIT that squeezes out further speedups for free.

By the end of this notebook you will be able to:
- Understand numeric precision trade-offs (FP32 vs FP16 vs BF16)
- Write a training loop with Automatic Mixed Precision (AMP)
- Benchmark the speedup of AMP + `torch.compile`


## 1. Numeric Precision Refresher

| Format | Bits | Range | Notes |
|--------|------|-------|-------|
| **FP32** (float32) | 32 | ±3.4 × 10³⁸ | Default PyTorch dtype — safest |
| **FP16** (float16) | 16 | ±65504 | Fast on GPU; can overflow/underflow |
| **BF16** (bfloat16) | 16 | ±3.4 × 10³⁸ | Same range as FP32, fewer mantissa bits; preferred on A100/H100 |

**Key insight:** Most of forward/backward pass arithmetic is safe in FP16/BF16.  
Only a few operations (loss scaling, batch norm stats) need FP32.  
AMP selects the right dtype *automatically* per operation.


In [ ]:
import torch
import torch.nn as nn
import time

# Memory footprint comparison
for dtype, name in [(torch.float32,'FP32'), (torch.float16,'FP16'), (torch.bfloat16,'BF16')]:
    t = torch.randn(1000, 1000, dtype=dtype)
    print(f"{name}: {t.element_size()} bytes/element → {t.nbytes/1024:.1f} KB for 1000×1000 tensor")

print()
print(f"FP16 memory saving vs FP32: {1 - 2/4:.0%}")
print(f"BF16 memory saving vs FP32: {1 - 2/4:.0%}")
print()
# Check which formats this GPU supports
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")


## 2. Automatic Mixed Precision (AMP)

PyTorch's `torch.cuda.amp` module handles the dtype switching automatically.

### The two components

**`autocast`** — wraps the forward pass; ops run in FP16/BF16 where safe:
```python
with torch.autocast(device_type='cuda', dtype=torch.float16):
    output = model(inputs)
    loss = criterion(output, targets)
```

**`GradScaler`** — prevents FP16 gradient underflow by scaling the loss up before backward,  
then unscaling before the optimizer step:
```python
scaler = torch.cuda.amp.GradScaler()
scaler.scale(loss).backward()
scaler.step(optimizer)
scaler.update()
```


In [ ]:
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

torch.manual_seed(42)

# Dataset
transform = transforms.Compose([transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])
train_ds = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)

# Simple CNN model
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128,256,3,padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(256*8*8, 512), nn.ReLU(), nn.Linear(512, 10))
    def forward(self, x): return self.classifier(self.features(x))

def train_epoch(model, loader, optimizer, criterion, use_amp=False, scaler=None):
    model.train()
    total_loss = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        if use_amp:
            with torch.autocast(device_type=device.type, dtype=torch.float16):
                loss = criterion(model(imgs), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

criterion = nn.CrossEntropyLoss()
print("Training setup ready. Run the benchmark cell below.")
print(f"Training on: {device}")


## 3. Benchmark: FP32 vs AMP vs AMP + torch.compile

`torch.compile()` (PyTorch ≥ 2.0) compiles the model into optimised machine code using TorchDynamo + TorchInductor.

| Mode | What it does |
|------|-------------|
| `torch.compile(model)` | Default — good balance of compile time and runtime |
| `torch.compile(model, mode='reduce-overhead')` | Reduces Python overhead — best for small batches |
| `torch.compile(model, mode='max-autotune')` | Longest compile, best throughput — use for production |


In [ ]:
import gc

results = {}
EPOCHS = 3

configs = [
    ('FP32 (baseline)',       False, False),
    ('AMP (FP16)',             True,  False),
    ('AMP + torch.compile',   True,  True ),
]

for name, use_amp, use_compile in configs:
    torch.manual_seed(42)
    model = SimpleCNN().to(device)
    if use_compile and device.type == 'cuda':
        model = torch.compile(model)   # ← one line!
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scaler = torch.cuda.amp.GradScaler() if use_amp and device.type == 'cuda' else None

    # Warm-up
    _ = train_epoch(model, train_loader, optimizer, criterion, use_amp, scaler)

    start = time.time()
    for _ in range(EPOCHS):
        loss = train_epoch(model, train_loader, optimizer, criterion, use_amp, scaler)
    elapsed = time.time() - start

    mem = torch.cuda.max_memory_allocated() / 1e9 if device.type == 'cuda' else 0
    results[name] = {'time': elapsed, 'mem_gb': mem, 'loss': loss}
    torch.cuda.reset_peak_memory_stats()
    gc.collect()
    print(f"[{name}]  {EPOCHS} epochs: {elapsed:.1f}s | Peak GPU mem: {mem:.2f} GB | Loss: {loss:.4f}")

# Summary
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
names = list(results.keys())
times = [results[n]['time'] for n in names]
mems  = [results[n]['mem_gb'] for n in names]
colors = ['#e74c3c', '#3498db', '#2ecc71']
axes[0].bar(names, times, color=colors); axes[0].set(title=f'Training Time ({EPOCHS} epochs)', ylabel='Seconds')
axes[1].bar(names, mems,  color=colors); axes[1].set(title='Peak GPU Memory', ylabel='GB')
for ax in axes:
    ax.grid(True, axis='y', alpha=0.3)
    for bar in ax.patches:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()


## 4. Exercises

1. **Memory experiment**: Add `torch.cuda.memory_summary()` after the FP32 run and after the AMP run. How much GPU memory is saved?
2. **BF16 vs FP16**: Modify the benchmark to also test `dtype=torch.bfloat16` in `autocast`. Which is faster on your GPU?
3. **compile modes**: Replace `torch.compile(model)` with `torch.compile(model, mode='reduce-overhead')`. Does it change the benchmark results?
